# "THE POUR IS RIGHT" - Wine Pricer Capstone

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 2: Data Pre-processing

Today an LLM rewrites every wine into a short, standard summary. This follows `week6/day2.ipynb`.

Ed needed this step because Amazon pages are long and messy. Our wine text is already tidy, so the rewrite does two other jobs:

1. **It removes price clues.** Some reviews give the answer away: 56 of our 22,000 quote a price ("a no-complaints kind of \$10 wine"), and 651 use words like "bargain", "value" or "price". A model trained on those would learn to read the answer instead of judging the wine.
2. **It makes every wine look the same.** One fixed format, with the same fields in the same order, is easier for the small open-source model we fine-tune in week 7.

## Costs

We use `gpt-oss-20b` on Groq through its Batch API, as Ed did. Ed's lite run cost under \$1, and our texts are shorter than his. A cell below measures the cost of one call before you spend anything on the full run.

In [ ]:
import re
import json
from dotenv import load_dotenv
from huggingface_hub import HfApi
from litellm import completion
from pricer.items import Item
from pricer.batch import Batch, SYSTEM_PROMPT, MODEL

load_dotenv(override=True)

In [ ]:
username = HfApi().whoami()["name"]
dataset = f"{username}/wines_raw_lite"

train, val, test = Item.from_hub(dataset)
items = train + val + test

print(f"Loaded {len(items):,} wines from {dataset}")
print(items[0])

In [ ]:
# Give every item an id, so we can match each batch result back to its wine

for index, item in enumerate(items):
    item.id = index

## The prompt

The prompt lives in `pricer/batch.py`, so the single test calls below and the full batch run use exactly the same instructions.

It keeps the points score. On day 3 we will test how much the models depend on it.

In [ ]:
print(SYSTEM_PROMPT)

In [ ]:
print(items[1007].full)

## Try one wine first

This is a normal (non-batch) call through LiteLLM, like Ed's. It prints the tokens used and the cost, so we can estimate the whole run before spending anything on it.

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model=f"groq/{MODEL}", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
cost = response._hidden_params["response_cost"]
print(f"Cost: {cost*100:.4f} cents")
print(f"Estimate for all {len(items):,} wines at this price: ${cost*len(items):.2f}")

The estimate above is at normal prices. Groq charges less for batch jobs, so the real cost of the full run should come in under it.

## Does it remove the price?

Here is a wine whose review mentions a dollar amount. The summary should keep the tasting notes and drop the price.

In [ ]:
price_mention = re.compile(r"\$\s?\d")

leaky = next(item for item in items if price_mention.search(item.full))
print(f"Actual price: ${leaky.price}\n")
print(leaky.full)

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": leaky.full}]
response = completion(messages=messages, model=f"groq/{MODEL}", reasoning_effort="low")
print("\nSUMMARY:\n")
print(response.choices[0].message.content)

## Optional: the same rewrite on a local model

Ed tried `llama3.2` on Ollama here. This uses `qwen3.5` instead. On this laptop it runs on the CPU at about 2 tokens per second, so one rewrite can take several minutes, and 22,000 would take weeks. That is why we use Groq for the real run.

Set `RUN_LOCAL_MODEL = True` to try it once.

In [ ]:
RUN_LOCAL_MODEL = False

if RUN_LOCAL_MODEL:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
    response = completion(messages=messages, model="ollama/qwen3.5:latest", api_base="http://localhost:11434", max_tokens=200)
    print(response.choices[0].message.content)

## The Batch API

Instead of 22,000 separate calls, we write the requests into JSONL files (one JSON request per line), upload them, and Groq works through them in the background, within 24 hours and usually much sooner.

Each line carries a `custom_id`, the item's id, so we can match each answer back to its wine. Here is the line for the first wine:

In [ ]:
print(json.dumps(json.loads(Batch(items, 0, 1).make_jsonl(items[0])), indent=2))

## Run the whole batch

The `Batch` class in `pricer/batch.py` is Ed's, adapted for wine. It:

- splits the items into groups of 1,000 (22 batches)
- writes a JSONL file for each, uploads it, and starts a batch job
- lets us check on them and collect the results when they are done

Ed's notebook first sent the first 1,000 items by hand, then sent everything again with the class, so those 1,000 were paid for twice. Here we only send each wine once.

**The next cell spends money.** Run it once.

In [ ]:
Batch.create(items)
Batch.run()
Batch.save()

## Collect the results

Run the next cell every few minutes until it says `Finished 22 of 22 batches`. It shows the status of the batches still waiting.

If the kernel restarts while you wait, don't run `Batch.run()` again, because that would pay for everything a second time. Instead, re-run the first four cells (imports, load, ids) and then the recovery cell below.

In [ ]:
Batch.fetch()
Batch.save()

In [ ]:
# RECOVERY ONLY - use this after a kernel restart instead of Batch.run()
# It reloads the submitted batches from batches.pkl and re-applies any results already downloaded

# Batch.load(items)
# for batch in Batch.batches:
#     if batch.done:
#         batch.apply_output()
# Batch.fetch()

## Check the results

Some requests in a batch can fail. We find any wine without a summary and rewrite those with normal calls.

In [ ]:
missing = [item for item in items if not item.summary]
print(f"{len(missing)} wines have no summary")

In [ ]:
for item in missing:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}]
    response = completion(messages=messages, model=f"groq/{MODEL}", reasoning_effort="low")
    item.summary = response.choices[0].message.content

print(f"{len([item for item in items if not item.summary])} wines still have no summary")

In [ ]:
# Did any price mentions survive the rewrite?

price_words = re.compile(r"\$\s?\d|\b(bargain|value|steal|price|priced|pricey|expensive|inexpensive|affordable|cheap)\b", re.I)

before = sum(1 for item in items if price_words.search(item.full))
after = [item for item in items if price_words.search(item.summary)]
print(f"Wines with a price clue: {before} before the rewrite, {len(after)} after")

for item in after[:5]:
    print("\n" + item.summary)

In [ ]:
print(items[0].summary)
print()
print(items[1000].summary)

## Push the final dataset to the hub

Ed removed the `full` text before pushing. We keep it, so on day 3 we can compare models trained on the original text with models trained on the summaries.

We only remove the `id`, which was just for matching batch results.

In [ ]:
# Second pass: rewrite the wines whose summary still has a price clue, with a stricter prompt

STRICT_PROMPT = SYSTEM_PROMPT + """
This is a second attempt, because the first summary mentioned price.
Your summary must not contain a dollar amount or any of these words: bargain, value, steal, price, priced, pricey, expensive, inexpensive, affordable, cheap.
Leave out any comparison with other wines by price."""

def has_price_clue(item):
    # Skip the first line: it is the wine's name, which can contain words like "Price Vineyard"
    body = item.summary.split("\n", 1)[-1]
    return price_words.search(body)

for attempt in range(2):
    leaky = [item for item in items if has_price_clue(item)]
    print(f"Attempt {attempt + 1}: {len(leaky)} summaries to rewrite")
    for item in leaky:
        messages = [{"role": "system", "content": STRICT_PROMPT}, {"role": "user", "content": item.full}]
        response = completion(messages=messages, model=f"groq/{MODEL}", reasoning_effort="low")
        item.summary = response.choices[0].message.content

remaining = [item for item in items if has_price_clue(item)]
print(f"{len(remaining)} summaries still have a price clue")
for item in remaining[:5]:
    print("\n" + item.summary)


In [ ]:
for item in items:
    item.summary = "\n".join(line.strip() for line in item.summary.strip().split("\n"))
    item.id = None

lite = f"{username}/wines_lite"

train = items[:20_000]
val = items[20_000:21_000]
test = items[21_000:]

Item.push_to_hub(lite, train, val, test)
print(f"Pushed to https://huggingface.co/datasets/{lite}")

In [ ]:
import json

with open("wines_lite_backup.jsonl", "w", encoding="utf-8") as f:
    for item in items:
        f.write(json.dumps(item.model_dump()) + "\n")

print(f"Backed up {len(items):,} wines")


In [ ]:
# Final clean-up, backup and push

import json
from huggingface_hub.utils import reset_sessions

for item in items:
    lines = [line.strip() for line in item.summary.split("\n") if line.strip()]
    lines = [f"Points: {item.points}" if line.startswith("Points:") else line for line in lines]
    item.summary = "\n".join(lines)
    item.id = None

with open("wines_lite_backup.jsonl", "w", encoding="utf-8") as f:
    for item in items:
        f.write(json.dumps(item.model_dump()) + "\n")

lite = f"{username}/wines_lite"

train = items[:20_000]
val = items[20_000:21_000]
test = items[21_000:]

reset_sessions()
Item.push_to_hub(lite, train, val, test)
print(f"Pushed to https://huggingface.co/datasets/{lite}")
